# Amortized Inference for the Fröhlich NLME Models
## Creation of Synthetic Data

**(model from Fröhlich et. al. 2018)**

In [ ]:
# load necessary packages
import numpy as np
import pandas as pd

# for plots
import matplotlib.pyplot as plt
from scipy.stats import lognorm

# estimating parameters
from sklearn.covariance import empirical_covariance

## Load the ODE model

In [ ]:
# specify which model to use
model_name = ['fröhlich-small', 'fröhlich-large', 'fröhlich-sde'][0]

print('using model =', model_name)
if model_name == 'fröhlich-small':
    from models.froehlich_model_small import FroehlichModelSmall
    model = FroehlichModelSmall()

    # mean of the variables (not of the normal distribution!)
    delta = 0.8  # np.log(2)/(0.8) # per hour
    gamma = 0.03  # np.log(2)/(22.8) # per hour
    k_m0_scale = 502  # a.u/h
    t_0 = 0.9  # in hours
    offset = 8  # a.u
    sigma = np.sqrt(0.001)  # must be positive
    theta_population = np.array([delta, gamma, k_m0_scale, t_0, offset, sigma])
    # covariance of random effects
    cov = np.diag(np.array([1, 1, 1000, 0.1, 0, 0])) # variances of the variables
elif model_name == 'fröhlich-large':
    from models.froehlich_model_large import FroehlichModelLarge
    model = FroehlichModelLarge()

    # mean of the variables (not of the normal distribution!)
    delta1_m0=1.2 # [h]
    delta2=0.6  # [h]
    e0_m0=0.85  #
    k2_m0_scale=1e6  # [a.u/h]
    k2=1.9  # [1/h]
    k1_m0=5.5  # [1/h]
    r0_m0=1e-1  # [1]
    gamma=0.01  # [1/h]
    t_0=0.9  # [h]
    offset=8  # [a.u]
    sigma=np.sqrt(0.001)
    theta_population = np.array([delta1_m0, delta2, e0_m0, k2_m0_scale, k2,
                                 k1_m0, r0_m0, gamma, t_0, offset, sigma])
    # covariance of random effects
    cov = np.diag(np.array([1.1, 0.4, 0.5, 10, 2, 100, 0.1, 0.01, 0.5, 0, 0])) # variances of the variables
elif model_name == 'fröhlich-sde':
    from models.froehlich_model_sde import FroehlichModelSDE
    model = FroehlichModelSDE()

    # mean of the variables (not of the normal distribution!)
    delta = 0.8  # np.log(2)/(0.8) # per hour
    gamma = 0.03  # np.log(2)/(22.8) # per hour
    k=1.44
    m0=300
    scale=2.12
    t_0 = 0.9  # in hours
    offset = 8  # a.u
    sigma = np.sqrt(0.001)  # must be positive
    theta_population = np.array([delta, gamma, k, m0, scale, t_0, offset, sigma])
    # covariance of random effects
    cov = np.diag(np.array([1, 1, 2, 5, 0, 0.1, 0, 0])) # variances of the variables
else:
    raise NotImplementedError('model not implemented')

simulator = model.build_simulator()
model.print_and_plot_example()

In [ ]:
# create lists with names of parameters
pop_param_names = ['pop-' + name for name in model.param_names]
var_param_names = ['var-' + name for name in model.param_names]
full_param_names = pop_param_names + var_param_names

## Creating Synthetic Data

We sample cell-specific parameters from population parameters and create synthetic observations. 

- all parameters are log-normally distributed
- offset and sigma do not vary over the data set

In [ ]:
# offset and sigma are fixed
def create_synthetic_data(n_cells: int,
                          population_param: np.ndarray,
                          cov_full: np.ndarray,
                          n_obs: int,
                          seed: int = 0):
    np.random.seed(seed)
    
    params_with_var = len(population_param)-2
    
    # cell-specific parameters with mixed effects
    mean = population_param[:params_with_var]
    cov = np.diag(np.diag(cov_full)[:params_with_var])

    # convert to log-normal mean and cov
    log_mean = np.log(mean**2 / np.sqrt(mean**2 + np.diag(cov)))
    log_cov = np.log(1 + np.diag(cov) / (mean**2))

    # cell-specific parameters with fixed effects
    offset = population_param[params_with_var]
    sigma = population_param[params_with_var+1]

    # all parameters except offset and noise are log-normal, those are fixed (and log) per experiment
    log_mean = np.concatenate((log_mean, np.array([np.log(offset), np.log(sigma)])), axis=0)
    log_cov = np.concatenate((log_cov, np.array([0, 0])), axis=0)

    # sample
    cell_param_log = np.random.multivariate_normal(log_mean, np.diag(log_cov), n_cells)

    # delete unreasonable params
    cell_param_log = cell_param_log[(cell_param_log[:, 0]-cell_param_log[:, 1]) != 0] # delta != gamma

    # simulate cells for parameters
    obs_data = simulator(cell_param_log)['sim_data'].reshape((n_cells, n_obs))
    
    # true parameter vector
    pop_params = np.concatenate((log_mean, log_cov), axis=0)

    return cell_param_log, obs_data, pop_params

In [ ]:
def get_sample_parameter_estimate(cell_param_log: np.ndarray):
    # estimate sample population parameters (will differ more for smaller samples)
    true_param_mean = np.mean(cell_param_log, axis=0) 
    true_param_cov_diag = np.diag(empirical_covariance(cell_param_log))
    true_param_sample = np.concatenate((true_param_mean, true_param_cov_diag))
    # compute mode of log-normal distribution
    mode_param_sample = np.exp(true_param_mean - true_param_cov_diag)
    return true_param_sample, mode_param_sample

In [ ]:
# create synthetic data
n_cells = [50, 100, 200, 500, 1000, 5000, 10000]

# create data and true parameters
cell_param_log, obs_data, true_pop_params = create_synthetic_data(n_cells[-1], theta_population, cov, model.n_obs)
true_mode = np.exp(true_pop_params[:model.n_params]-true_pop_params[model.n_params:])

# parameters estimated from sample (only those can be recovered)
true_pop_param_sample = np.zeros((len(n_cells), model.n_params*2))
mode_param_sample = np.zeros((len(n_cells), model.n_params))
for i, cells in enumerate(n_cells):
    true_pop_param_sample[i], mode_param_sample[i] = get_sample_parameter_estimate(cell_param_log[:cells])

In [ ]:
# save data
t_points = np.linspace(start=1/6, stop=30, num=model.n_obs, endpoint=True)

# save into csv data
df_obs = pd.DataFrame(np.exp(obs_data.reshape(n_cells[-1], model.n_obs).T), index=(t_points*60*60).round(0).astype(int))
if model_name == 'fröhlich-small':
    pass
    #df_obs.to_csv("data/synthetic/data_random_cells.csv")
elif model_name == 'fröhlich-large':
    df_obs.to_csv("data/synthetic/data_random_cells_large_model.csv")
elif model_name == 'fröhlich-sde':
    df_obs.to_csv("data/synthetic/data_random_cells_sde_model.csv")
else:
    raise NotImplementedError

df_params = pd.DataFrame(cell_param_log, columns=model.log_param_names)
if model_name == 'fröhlich-small':
    pass
    #df_params.to_csv("data/synthetic/synthetic_individual_cell_params.csv")
elif model_name == 'fröhlich-large':
    df_params.to_csv("data/synthetic/synthetic_individual_cell_params_large_model.csv")
elif model_name == 'fröhlich-sde':
    df_params.to_csv("data/synthetic/synthetic_individual_cell_params_sde_model.csv")
else:
    raise NotImplementedError

In [ ]:
# plot synthetic data
plt.figure(figsize=(8, 5), tight_layout=True)
for i in range(n_cells[2]):
    plt.plot(t_points, np.exp(obs_data[i]), color='grey', alpha=0.3)

plt.xlabel('$t$ (in hours)')
plt.ylabel('fluorescence intensity (in a.u.)')
plt.yscale('log')
plt.title(f'Single Cell Expression')
plt.show()

In [ ]:
from numba import njit
from pypesto import FD, Objective, Problem, optimize, engine
from functools import partial
import warnings
from heatmap import corrplot

In [ ]:
simulator_noise_free = model.build_simulator(with_noise=False)

In [ ]:
@njit
def log_likelihood_inner(measurement: np.ndarray, simulation: np.ndarray, sigma: float, n_points: int):
    # compute the log-likelihood
    dif = np.sum((measurement - simulation) ** 2)
    llh = n_points * np.log(2 * np.pi * sigma ** 2) + dif / (sigma ** 2)
    return -0.5 * llh


def log_likelihood(param: np.ndarray, measurement: np.ndarray):
    # simulate data
    n_points = measurement.size
    p_samples = param.reshape((1, param.size))

    with warnings.catch_warnings():
        # some samples will generate errors because of overflow
        warnings.filterwarnings("ignore")
        simulation = simulator_noise_free(p_samples)['sim_data'].reshape(n_points)

        # compute the log-likelihood
        #delta, gamma, k_m0_scale, t_0, offset, sigma = np.exp(param_samples[ns, :])
        llh = log_likelihood_inner(measurement=measurement, simulation=simulation,
                                   sigma=np.exp(param[-1]), n_points=n_points)
    return -llh

In [ ]:
param_estimates = np.zeros((50, model.n_params))

for cell_id, single_cell in enumerate(obs_data[:param_estimates.shape[0]]):
    pesto_objective = FD(obj=Objective(fun=partial(log_likelihood, measurement=single_cell),
                                    x_names=model.log_param_names))

    lb = cell_param_log[cell_id, :model.n_params] - 2.58
    ub = cell_param_log[cell_id, :model.n_params] + 2.58

    # create pypesto problem
    pesto_problem = Problem(objective=pesto_objective,
                            lb=lb, ub=ub,
                            x_names=model.log_param_names)
    #pesto_problem.print_parameter_summary()

    result = optimize.minimize(
                problem=pesto_problem,
                optimizer=optimize.ScipyOptimizer(),
                n_starts=100,
                engine=engine.MultiProcessEngine(8))
    param_estimates[cell_id] = result.optimize_result.as_dataframe()['x'][0]

In [ ]:
change_modes_idx = param_estimates[:, 0] < param_estimates[:, 1]
temp = param_estimates[change_modes_idx, 0].copy()
param_estimates[change_modes_idx, 0] = param_estimates[change_modes_idx, 1]
param_estimates[change_modes_idx, 1] = temp
param_estimates

In [ ]:
sample_mean = np.mean(cell_param_log[:param_estimates.shape[0], :model.n_params], axis=0)
sample_cov = np.cov(cell_param_log[:param_estimates.shape[0], :model.n_params].T)
real_mean_sim = simulator_noise_free(sample_mean[np.newaxis])['sim_data'].reshape(180)

estimated_mean = np.mean(param_estimates, axis=0)
estimated_cov = np.cov(param_estimates.T)
mean_sim = simulator_noise_free(estimated_mean[np.newaxis])['sim_data'].reshape(180)

In [ ]:
error_cov = (sample_cov-estimated_cov)
e_min = np.min(error_cov)
e_max = np.max(error_cov)
cov_df = pd.DataFrame(error_cov, columns=model.param_names, index=model.param_names)
ax = corrplot(cov_df, color_range=[e_min, e_max],  size_scale=500)
print(error_cov.sum())

In [ ]:
# plot synthetic data
plt.figure(figsize=(8, 5), tight_layout=True)
for single_cell_param in param_estimates:
    single_cell = simulator_noise_free(single_cell_param[np.newaxis])['sim_data'].reshape(180)
    plt.plot(t_points, np.exp(single_cell), color='red', alpha=0.3)
for single_cell in obs_data[:param_estimates.shape[0]]:
    plt.plot(t_points, np.exp(single_cell), color='grey', alpha=0.3)
#plt.plot(t_points, np.exp(real_mean_sim), color='red', linewidth=2, label='simulation of parameter mean')
#plt.plot(t_points, np.exp(mean_sim), color='green', linewidth=2, label='mean of simulations')

plt.xlabel('$t$ (in hours)')
plt.ylabel('fluorescence intensity (in a.u.)')
plt.yscale('log')
plt.title(f'Single Cell Expression')
#plt.legend()
plt.savefig('plots/synthetic_data_single_fit.png', dpi=600)
plt.show()

In [ ]:
# plot synthetic data
plt.figure(figsize=(8, 5), tight_layout=True)
for single_cell in obs_data[:param_estimates.shape[0]]:
    plt.plot(t_points, np.exp(single_cell), color='grey', alpha=0.3)
plt.plot(t_points, np.exp(real_mean_sim), color='red', linewidth=2, label='simulation of parameter mean')
plt.plot(t_points, np.exp(mean_sim), color='green', linewidth=2, label='mean of simulations')

plt.xlabel('$t$ (in hours)')
plt.ylabel('fluorescence intensity (in a.u.)')
plt.yscale('log')
plt.title(f'Single Cell Expression')
plt.legend()
#plt.savefig('plots/synthetic_data_means.png', dpi=600)
plt.show()

In [ ]:
all_params = np.concatenate(([true_pop_params], true_pop_param_sample)).round(5)
all_modes = np.concatenate(([true_mode], mode_param_sample)).round(5)

print(f'Parameters for samples of the population (log-normal distribution)')
df_sample_pop_parameters = pd.DataFrame(all_params, columns=full_param_names, index=['true']+n_cells)
display(df_sample_pop_parameters)
if model_name == 'fröhlich-small':
    df_sample_pop_parameters.to_csv(f'data/synthetic/sample_pop_parameters.csv')
elif model_name == 'fröhlich-large':
    df_sample_pop_parameters.to_csv(f'data/synthetic/sample_pop_parameters_large_model.csv')
elif model_name == 'fröhlich-sde':
    df_sample_pop_parameters.to_csv(f'data/synthetic/sample_pop_parameters_sde_model.csv')
else:
    raise NotImplementedError

print('true modes')
df_modes = pd.DataFrame(all_modes, columns=model.param_names, index=['true']+n_cells)
display(df_modes)

In [ ]:
# plot posteriors for each parameter individually
figure, axis = plt.subplots(nrows=1, ncols=len(model.param_names), figsize=(15,5))

for j, params in enumerate(all_params):
    for i, name in enumerate(model.param_names):

        x = np.linspace(np.exp(model.prior_mean[i] - 1.98*model.prior_std[i]),
                        np.exp(model.prior_mean[i] + 1.98*model.prior_std[i]), 10000)

        # Set parameters for the log-normal distribution
        mu = params[i]
        sigma = params[len(model.param_names)+i]

        # Create a log-normal distribution with the given parameters
        log_norm = lognorm(sigma, scale=np.exp(mu))

        # Plot the density function
        axis[i].plot(x, log_norm.pdf(x), label=f'cells {df_sample_pop_parameters.index[j]}')
        axis[i].set_title(name)
        axis[i].set_xscale('log')

    if model_name == 'fröhlich-sde':
        axis[4].axvline(np.exp(params[4]), linestyle='-.')

    axis[len(model.param_names)-2].axvline(np.exp(params[len(model.param_names)-2]), linestyle='-.')
    axis[len(model.param_names)-1].axvline(np.exp(params[len(model.param_names)-1]), linestyle='-.')

    break

plt.legend()
if model_name == 'fröhlich-small':
    plt.savefig('plots/synthetic_log_normal_distributions.png')
elif model_name == 'fröhlich-large':
    plt.savefig('plots/synthetic_log_normal_distributions_large_model.png')
elif model_name == 'fröhlich-sde':
    plt.savefig('plots/synthetic_log_normal_distributions_sde_model.png')
else:
    raise NotImplementedError
plt.show()